In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras.preprocessing.text import Tokenizer

# A. DATA PREPARATION
text = """I love machine learning
I love deep learning
machine learning is interesting
deep learning is powerful"""

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

data = tokenizer.texts_to_sequences([text])[0]
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print("Vocabulary:", word_index)

# B. GENERATE TRAINING DATA
X, y = [], []
window = 1

for i in range(window, len(data) - window):
    context = data[i-window:i] + data[i+1:i+window+1]
    X.append(context)
    y.append(data[i])

X = np.array(X)
y = tf.keras.utils.to_categorical(y, vocab_size)

print("\nTraining Data:")
print("X =", X)

# C. TRAIN CBOW MODEL
model = Sequential([
    Embedding(vocab_size, 10),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X, y, epochs=200, verbose=0)

# D. OUTPUT / PREDICTION
reverse_word = {v: k for k, v in word_index.items()}

context = ["love", "learning"]
input_data = np.array([[word_index[w] for w in context]])

prediction = model.predict(input_data, verbose=0)
predicted_id = np.argmax(prediction)

print("\nContext Words:", context)
print("Predicted Word:", reverse_word[predicted_id])


Vocabulary: {'learning': 1, 'i': 2, 'love': 3, 'machine': 4, 'deep': 5, 'is': 6, 'interesting': 7, 'powerful': 8}

Training Data:
X = [[2 4]
 [3 1]
 [4 2]
 [1 3]
 [2 5]
 [3 1]
 [5 4]
 [1 1]
 [4 6]
 [1 7]
 [6 5]
 [7 1]
 [5 6]
 [1 8]]

Context Words: ['love', 'learning']
Predicted Word: deep
